<div align="center">

<a href="https://colab.research.google.com/github/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-07/NB07_ml_workflow_regression.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 07: Learning from Data: The Machine Learning Workflow and Regression

**Artificial Intelligence Applications in Engineering (MUH-920 Mühendislikte Yapay Zeka Uygulamaları)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)<br>[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

<sub>Content licensed under CC BY 4.0, code under MIT. This course is updated in line with current developments in the field. Last update: October 2026.</sub>

</div>

## About this notebook

The notebook loads the UCI concrete compressive strength data [1], explores it with pandas, derives the water-binder ratio, and compares a mean baseline, linear and ridge regression, k-nearest neighbours, a regression tree, a random forest and gradient boosting with five-fold cross-validation [2]. The best model is tested once on held-out data with parity and residual plots, a deliberate leakage experiment shows how scores inflate, and a one-line switch reruns everything on the dataset of another department.

This notebook is the hands-on part of Week 7. It opens with Python step 7: Tables with pandas and honest evaluation with scikit-learn and continues with the sections listed in the table of contents. The concepts are explained on the [lecture page](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-07/lecture.html) and in the [PDF notes](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-07/Week07_Lecture_Notes.pdf). The [interactive lab](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-07/lab.html) holds the simulation, the self-assessment and the reflection, and the [week overview](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-07/README.md) gives the learning outcomes, the study path, the discipline challenges and the tasks.

## How to use this notebook

Run the cells in order with Shift+Enter. No earlier Python experience is assumed: Python step 7 teaches the Python of the week first, and every further piece of Python appears in a cell marked **Python move**, which explains the syntax right where it is first needed. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: Open them only after a genuine attempt. Cells that download public data need an internet connection. When a source is unreachable, the notebook falls back to a documented synthetic stand-in so that every later cell still runs.

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_validate, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = abs(float(value) - float(expected)) <= tol * max(1.0, abs(float(expected)))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] {'Correct.' if ok else 'Not yet. Check the logic and run again.'}")

## Python step 7: Tables with pandas and honest evaluation with scikit-learn

This step teaches the Python of the week with the week's own example. Read each explanation, run the cell below it and compare the output with the text; then change a value and predict the new result before running the cell again. A quick check and short exercises with immediate feedback close the step. The [Python path](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/PYTHON_PATH.md) shows how the fourteen steps of the course build on each other.

### Tables as DataFrames

Most engineering data arrive as tables, with one row per specimen, test or time and one column per variable. The pandas library represents such a table as a DataFrame [3]. A DataFrame can be built from a dictionary of columns, and `shape`, `head` and `describe` give a first view. The values below are invented for practice and only loosely follow the behaviour of real concrete; the hands-on sections below use the measured Concrete Compressive Strength dataset [1].

In [ ]:
import pandas as pd

mixes = pd.DataFrame({
    "cement_kg": [380, 300, 450, 320, 400, 280, 350, 420],
    "water_kg": [175, 180, 160, 190, 170, 185, 165, 150],
    "age_days": [28, 28, 28, 7, 90, 28, 7, 28],
    "strength_MPa": [44.1, 33.9, 55.2, 21.3, 52.6, 29.8, 30.4, 58.7],
})
print(mixes.shape)
print(mixes.head(3))

### Selecting, computing and filtering

A column is selected with its name in square brackets and returns a Series, a single labelled column. A list of names selects several columns. New columns come from arithmetic on existing ones, such as the water-cement ratio, the main factor of concrete strength. A condition on a column produces a boolean Series, and using it as an index keeps the matching rows. Conditions combine with `&` for and and `|` for or, each condition in its own parentheses.

In [ ]:
mixes["wc_ratio"] = mixes["water_kg"] / mixes["cement_kg"]
at28 = mixes[mixes["age_days"] == 28]
print(at28[["wc_ratio", "strength_MPa"]].round(3))
strong = mixes[(mixes["age_days"] == 28) & (mixes["strength_MPa"] > 40)]
print(len(strong), "mixes above 40 MPa at 28 days")
print(mixes.sort_values("wc_ratio")[["cement_kg", "wc_ratio"]].head(2))

### Reading files and missing values

`pd.read_csv` reads comma-separated files from a path or a web address. Empty fields become `NaN`, the marker of a missing value. `isna().sum()` counts the missing values of each column, and `dropna` removes incomplete rows. Whether removing, filling or modelling missing values is appropriate depends on why they are missing. The example reads a short text as if it were a file.

In [ ]:
import io

csv_text = """cement_kg,water_kg,age_days,strength_MPa
380,175,28,44.1
300,,28,33.9
450,160,28,
"""
table = pd.read_csv(io.StringIO(csv_text))
print(table.isna().sum())
print(table.dropna())

### The scikit-learn pattern

scikit-learn gives all its models the same interface [2]. A model object is created with its settings, `fit` learns from the inputs X and the targets y, and `predict` applies the model to new inputs. `train_test_split` holds back part of the data for testing, and `random_state` makes the split repeatable. With eight invented rows, the numbers below practise the syntax only; the workflow of this week explains how models are validated properly.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

X = mixes[["wc_ratio", "age_days"]]
y = mixes["strength_MPa"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
model = LinearRegression().fit(X_train, y_train)
print(model.coef_.round(2), round(float(model.intercept_), 2))
print(model.predict(X_test).round(1), y_test.to_numpy())

### Measuring errors

A model is judged by its errors on data it has not seen. The mean absolute error (MAE) averages the size of the errors in the unit of the target, here MPa, and the root mean squared error (RMSE) weights large errors more strongly. Both take one line of NumPy, and `sklearn.metrics` provides the same measures. A baseline that always predicts the mean strength of the training mixes shows whether the model has learned anything at all.

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

pred_test = model.predict(X_test)
err = y_test.to_numpy() - pred_test
print("MAE by hand:", round(float(np.mean(np.abs(err))), 2), "MPa")
print("MAE with scikit-learn:", round(float(mean_absolute_error(y_test, pred_test)), 2), "MPa")
print("RMSE:", round(float(np.sqrt(mean_squared_error(y_test, pred_test))), 2), "MPa")
baseline = np.full(len(y_test), y_train.mean())
print("MAE of the mean baseline:", round(float(mean_absolute_error(y_test, baseline)), 2), "MPa")

### Cross-validation and pipelines

A single split depends on which rows happen to land in the test set. K-fold cross-validation divides the data into k parts, trains k times and tests each time on a different part, which gives k error estimates instead of one. Preprocessing such as scaling must be learned from the training folds only; otherwise information from the test fold leaks into the model. A pipeline chains the preprocessing and the model, so that `cross_val_score` refits both inside every fold.

In [ ]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

folds = KFold(n_splits=4, shuffle=True, random_state=0)
pipe = make_pipeline(StandardScaler(), LinearRegression())
scores = -cross_val_score(pipe, X, y, cv=folds, scoring="neg_mean_absolute_error")
print(scores.round(2), "mean MAE:", round(float(scores.mean()), 2), "MPa")

scikit-learn treats larger scores as better, so it reports errors with a negative sign, and the minus in front turns them back into MAE values. With eight invented rows, the numbers serve to practise the pattern, which the hands-on sections below apply to the full dataset.

**Quick check.** A DataFrame df has the columns age and f. Which expression keeps the rows with age 28 and f above 40?

A. df[df["age"] == 28 and df["f"] > 40]  
B. df[(df["age"] == 28) & (df["f"] > 40)]  
C. df["age" == 28]  
D. df.keep(age=28, f=40)

<details><summary>Show the answer</summary>

**B.** pandas combines conditions element by element with & and |, and each condition needs its own parentheses. The keyword and cannot combine two Series.

</details>

### Exercises for Python step 7

**Exercise 7.1.** Compute the mean water-cement ratio of the mixes in `mixes` whose strength exceeds 45 MPa and store it in `py_wc_strong`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py7_mixes():
    import pandas as pd
    m = pd.DataFrame({"cement_kg": [380, 300, 450, 320, 400, 280, 350, 420], "water_kg": [175, 180, 160, 190, 170, 185, 165, 150],
                      "age_days": [28, 28, 28, 7, 90, 28, 7, 28], "strength_MPa": [44.1, 33.9, 55.2, 21.3, 52.6, 29.8, 30.4, 58.7]})
    m["wc_ratio"] = m["water_kg"] / m["cement_kg"]
    return m


def _py7_wc_strong_reference():
    m = _py7_mixes()
    return float(m.loc[m["strength_MPa"] > 45, "wc_ratio"].mean())

In [ ]:
py_wc_strong = None
check("Exercise 7.1", py_wc_strong, _py7_wc_strong_reference())

**Exercise 7.2.** Store the total number of missing values in `table` in `py_missing`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py7_missing_reference():
    return 2

In [ ]:
py_missing = None
check("Exercise 7.2", py_missing, _py7_missing_reference())

**Exercise 7.3.** Use `model` to predict the strength of a mix with a water-cement ratio of 0.40 at 28 days and store the number in `py_pred`. Build the input as a DataFrame with the same column names as `X`, for example `pd.DataFrame({"wc_ratio": [0.40], "age_days": [28]})`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py7_pred_reference():
    import pandas as pd
    from sklearn.linear_model import LinearRegression
    from sklearn.model_selection import train_test_split
    m = _py7_mixes()
    Xa, Xb, ya, yb = train_test_split(m[["wc_ratio", "age_days"]], m["strength_MPa"], test_size=0.25, random_state=0)
    reg = LinearRegression().fit(Xa, ya)
    return float(reg.predict(pd.DataFrame({"wc_ratio": [0.40], "age_days": [28]}))[0])

In [ ]:
py_pred = None
check("Exercise 7.3", py_pred, _py7_pred_reference())

**Exercise 7.4.** Store the MAE of the mean baseline on the test set in `py_baseline_mae` and compare it with the MAE of the model.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py7_baseline_mae_reference():
    import numpy as np
    from sklearn.model_selection import train_test_split
    m = _py7_mixes()
    Xa, Xb, ya, yb = train_test_split(m[["wc_ratio", "age_days"]], m["strength_MPa"], test_size=0.25, random_state=0)
    return float(np.mean(np.abs(yb.to_numpy() - ya.mean())))

In [ ]:
py_baseline_mae = None
check("Exercise 7.4", py_baseline_mae, _py7_baseline_mae_reference())

**Exercise 7.5.** Run the same cross-validation with `LinearRegression()` alone, without the scaler, and store the mean MAE in `py_cv_plain`. Why is the result the same as with the pipeline for this model?

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py7_cv_plain_reference():
    from sklearn.linear_model import LinearRegression
    from sklearn.model_selection import KFold, cross_val_score
    m = _py7_mixes()
    s = -cross_val_score(LinearRegression(), m[["wc_ratio", "age_days"]], m["strength_MPa"],
                         cv=KFold(n_splits=4, shuffle=True, random_state=0), scoring="neg_mean_absolute_error")
    return float(s.mean())

In [ ]:
py_cv_plain = None
check("Exercise 7.5", py_cv_plain, _py7_cv_plain_reference())

### A reusable loader for public datasets

**Python move: a reusable function with keyword arguments.** `load_uci` downloads a dataset from the UCI Machine Learning Repository with the `ucimlrepo` package [4] and returns the features as a pandas DataFrame and the target as a Series. The package is installed on first use. If the repository cannot be reached, the calling cell switches to a documented synthetic stand-in, so that every later cell still runs.

In [ ]:
import subprocess
import sys


def load_uci(uci_id, target=None, drop=(), onehot_targets=False):
    """Return (X, y) for a UCI dataset; target names a column of the targets or of the features."""
    try:
        from ucimlrepo import fetch_ucirepo
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--retries", "1", "--timeout", "15", "ucimlrepo"], check=False)
        from ucimlrepo import fetch_ucirepo
    data = fetch_ucirepo(id=uci_id).data
    X = data.features.copy()
    Y = data.targets.copy() if data.targets is not None else pd.DataFrame(index=X.index)
    if onehot_targets:                                   # several 0/1 target columns, one per class
        y = Y.idxmax(axis=1)
    else:
        if target is None:
            target = Y.columns[0]
        y = X.pop(target) if target in X.columns else Y[target]
    X = X.drop(columns=[c for c in drop if c in X.columns])
    return X, y.astype(float) if y.dtype.kind in "iuf" else y

## 1. The concrete dataset

The UCI dataset 165 contains 1030 concrete mixes with seven ingredients in kg per cubic metre, the age in days and the measured compressive strength in MPa [1, 5]. If the download fails, a synthetic stand-in with the same columns is generated from Abrams' water-cement rule and a strength-age curve [6]; its numbers are illustrative, not measurements.

**Python move: DataFrames.** `X.head()` shows the first rows, `X.shape` gives rows and columns, and `X.describe()` summarises every column.

In [ ]:
def concrete_stand_in(n=1030, seed=165):
    rng = np.random.default_rng(seed)
    cement = rng.uniform(100, 540, n)
    slag = np.where(rng.random(n) < 0.45, 0, rng.uniform(10, 360, n))
    ash = np.where(rng.random(n) < 0.55, 0, rng.uniform(20, 200, n))
    water = rng.uniform(125, 245, n)
    sp = np.where(rng.random(n) < 0.37, 0, rng.uniform(1, 32, n))
    coarse, fine = rng.uniform(800, 1145, n), rng.uniform(595, 990, n)
    age = rng.choice([3, 7, 14, 28, 56, 90, 180, 365], n, p=[.13, .12, .06, .41, .09, .13, .03, .03])
    wb = water / (cement + 0.8 * slag + 0.4 * ash)
    strength = 110 / 8.5 ** wb * age / (4 + 0.85 * age) * (1 + 0.004 * sp) + rng.normal(0, 4, n)
    X = pd.DataFrame({"Cement": cement, "Blast Furnace Slag": slag, "Fly Ash": ash, "Water": water, "Superplasticizer": sp,
                      "Coarse Aggregate": coarse, "Fine Aggregate": fine, "Age": age})
    return X, pd.Series(np.clip(strength, 2, 85), name="Concrete compressive strength")


try:
    X, y = load_uci(165)
    SOURCE = "UCI 165, Concrete Compressive Strength"
except Exception as error:
    print("Download failed, using the synthetic stand-in:", type(error).__name__)
    X, y = concrete_stand_in()
    SOURCE = "synthetic stand-in for UCI 165"
print(SOURCE, X.shape)
X.head()

In [ ]:
X.describe().T.round(1)

## 2. Exploration and a physically motivated feature

**Python move: selecting columns by pattern and adding columns.** The helper `col` finds a column whose name contains a word, which keeps the code robust to small differences in column names.

In [ ]:
def col(frame, word):
    return next(c for c in frame.columns if word.lower() in c.lower())


C_CEM, C_SLAG, C_ASH, C_WAT, C_AGE = (col(X, w) for w in ["cement", "slag", "ash", "water", "age"])
X["w_b"] = X[C_WAT] / (X[C_CEM] + X[C_SLAG] + X[C_ASH])        # water-binder ratio
fig, ax = plt.subplots(1, 3, figsize=(11, 3.2))
ax[0].hist(y, bins=30); ax[0].set_xlabel("strength (MPa)")
ax[1].scatter(X["w_b"], y, s=6, alpha=0.5); ax[1].set_xlabel("water-binder ratio"); ax[1].set_ylabel("strength (MPa)")
means = y.groupby(X[C_AGE]).mean()
ax[2].plot(means.index, means.values, "o-"); ax[2].set_xscale("log"); ax[2].set_xlabel("age (days)"); ax[2].set_ylabel("mean strength (MPa)")
plt.tight_layout(); plt.show()

### Your turn, exercise 1

Compute the mean water-binder ratio of the specimens tested at 28 days and store it in `wb_28`. Use a Boolean mask on the age column.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _wb_28_reference():
    return float(X.loc[X[C_AGE] == 28, "w_b"].mean())

In [ ]:
wb_28 = None   # your computation
check("Exercise 1", wb_28, _wb_28_reference(), tol=1e-6)

## 3. Split first, then compare models with cross-validation

**Python move: the scikit-learn interface.** Every model has `fit` and `predict`. `make_pipeline` chains scaling and a model, so the scaler is fitted inside each training fold only [2, 7].

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
MODELS = {
    "mean baseline": DummyRegressor(),
    "linear": make_pipeline(StandardScaler(), LinearRegression()),
    "ridge (alpha 10)": make_pipeline(StandardScaler(), Ridge(alpha=10)),
    "k-NN (k 7)": make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=7)),
    "tree (depth 8)": DecisionTreeRegressor(max_depth=8, random_state=0),
    "random forest": RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1),
    "gradient boosting": HistGradientBoostingRegressor(random_state=0),
}


def compare(models, X, y, folds=5):
    rows = []
    for name, model in models.items():
        cv = cross_validate(model, X, y, cv=KFold(folds, shuffle=True, random_state=0),
                            scoring=("neg_mean_absolute_error", "neg_root_mean_squared_error", "r2"), return_train_score=True)
        rows.append({"model": name, "CV MAE": -cv["test_neg_mean_absolute_error"].mean(),
                     "CV RMSE": -cv["test_neg_root_mean_squared_error"].mean(), "CV R2": cv["test_r2"].mean(), "train R2": cv["train_r2"].mean()})
    return pd.DataFrame(rows).set_index("model").round(3)


table = compare(MODELS, X_train, y_train)
table

### Your turn, exercise 2

Run five-fold cross-validation (shuffled, `random_state=0`) of a ridge pipeline with `alpha=100` on the training data and store the mean MAE in `ridge100_mae`. Is stronger regularisation better or worse here?

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _ridge100_mae_reference():
    m = make_pipeline(StandardScaler(), Ridge(alpha=100))
    cv = cross_validate(m, X_train, y_train, cv=KFold(5, shuffle=True, random_state=0), scoring="neg_mean_absolute_error")
    return float(-cv["test_score"].mean())

In [ ]:
ridge100_mae = None   # your computation
check("Exercise 2", ridge100_mae, _ridge100_mae_reference(), tol=1e-6)

## 4. One final test, with plots

The model with the lowest cross-validated MAE is refitted on all training data and evaluated once on the test set.

In [ ]:
best_name = table["CV MAE"].idxmin()
best = MODELS[best_name].fit(X_train, y_train)
pred = best.predict(X_test)
print(f"{best_name}: test MAE {mean_absolute_error(y_test, pred):.2f}, RMSE {mean_squared_error(y_test, pred) ** 0.5:.2f}, R2 {r2_score(y_test, pred):.3f}")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
ax[0].scatter(y_test, pred, s=10, alpha=0.7); lim = [0, max(y_test.max(), pred.max()) + 5]
ax[0].plot(lim, lim, "k--", lw=0.8); ax[0].set_xlabel("measured (MPa)"); ax[0].set_ylabel("predicted (MPa)"); ax[0].set_title(f"Parity plot, {best_name}")
ax[1].scatter(X_test[C_AGE], y_test - pred, s=10, alpha=0.7); ax[1].axhline(0, color="k", lw=0.8); ax[1].set_xscale("log")
ax[1].set_xlabel("age (days)"); ax[1].set_ylabel("residual (MPa)"); ax[1].set_title("Residuals against age")
plt.tight_layout(); plt.show()

## 5. A leakage experiment

A "strength class" in steps of 5 MPa is computed from the measured strength and added as a feature. Nothing in the cross-validation code warns about it; only the question of when the feature becomes known reveals the problem [8].

In [ ]:
X_leak = X_train.assign(strength_class=(y_train // 5) * 5)
leaky = compare({"linear with leaked feature": make_pipeline(StandardScaler(), LinearRegression())}, X_leak, y_train)
pd.concat([table.loc[["linear"]], leaky])

### Your turn, exercise 3

Fit a random forest (`n_estimators=200`, `random_state=0`) on the training data without the leaked feature and store its R-squared on the test set in `rf_test_r2`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _rf_test_r2_reference():
    rf = RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1).fit(X_train, y_train)
    return float(r2_score(y_test, rf.predict(X_test)))

In [ ]:
rf_test_r2 = None   # your computation
check("Exercise 3", rf_test_r2, _rf_test_r2_reference(), tol=1e-6)

## 6. The discipline switch

Set `CHOICE` to one of the keys below and run the cell. The same comparison runs on another engineering dataset [9, 10, 11, 12, 13, 14, 15]. Categorical columns are one-hot encoded and missing values are filled with column medians inside the training data. If a download fails, the concrete data are used again and a message says so.

In [ ]:
SWITCH = {
    "concrete (civil)": (165, None, ()),
    "power plant (mechanical, electrical)": (294, "PE", ()),
    "building heating load (civil, physics)": (242, "Y1", ()),
    "airfoil noise (automotive, mechanical)": (291, "scaled-sound-pressure", ()),
    "gas turbine NOx (environmental, chemical)": (551, "NOX", ("CO",)),
    "superconductors (physics, chemistry)": (464, "critical_temp", ()),
    "garment productivity (industrial, textile)": (597, "actual_productivity", ("date",)),
    "wine quality (food, chemistry)": (186, "quality", ()),
}
CHOICE = "power plant (mechanical, electrical)"
uid, target, drop = SWITCH[CHOICE]
try:
    Xd, yd = load_uci(uid, target=target, drop=drop)
    Xd = pd.get_dummies(Xd, drop_first=True).astype(float)
    if len(Xd) > 8000:                                   # keep the comparison quick on large datasets
        idx = Xd.sample(8000, random_state=0).index
        Xd, yd = Xd.loc[idx], yd.loc[idx]
except Exception as error:
    print(f"Could not load {CHOICE} ({type(error).__name__}); using the concrete data instead.")
    Xd, yd = X.drop(columns="w_b"), y
Xtr, Xte, ytr, yte = train_test_split(Xd, yd, test_size=0.2, random_state=42)
Xtr = Xtr.fillna(Xtr.median()); Xte = Xte.fillna(Xtr.median())
compare({k: MODELS[k] for k in ["mean baseline", "linear", "random forest", "gradient boosting"]}, Xtr, ytr)

## Next steps

Take the [self-assessment](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-07/lab.html#check) in the interactive lab, then answer the [reflection prompts](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-07/lab.html#reflect) and export the learning log. The [discipline challenges](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-07/README.md#discipline-challenges), the [weekly task](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-07/README.md#weekly-task) and the [research assignment](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-07/README.md#research-and-report-assignment-optional) are listed in the week overview, and [Week 8](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-08/README.md) starts with its overview.

## References

[1] Yeh, I.-C. (1998). *Concrete Compressive Strength [Dataset]. UCI Machine Learning Repository, ID 165*. <https://doi.org/10.24432/C5PK67>

[2] Pedregosa, F., Varoquaux, G., Gramfort, A., et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research*, *12*, 2825-2830.

[3] McKinney, W. (2010). Data structures for statistical computing in Python. In *Proceedings of the 9th Python in Science Conference* (pp. 56-61). <https://doi.org/10.25080/Majora-92bf1922-00a>

[4] Kelly, M., Longjohn, R., & Nottingham, K. (2026). *The UCI Machine Learning Repository*. <https://archive.ics.uci.edu>

[5] Yeh, I.-C. (1998). Modeling of strength of high-performance concrete using artificial neural networks. *Cement and Concrete Research*, *28*(12), 1797-1808. <https://doi.org/10.1016/S0008-8846(98)00165-3>

[6] Abrams, D. A. (1918). *Design of Concrete Mixtures (Bulletin 1)*. Structural Materials Research Laboratory, Lewis Institute, Chicago.

[7] Kohavi, R. (1995). A study of cross-validation and bootstrap for accuracy estimation and model selection. In *Proceedings of the 14th International Joint Conference on Artificial Intelligence* (pp. 1137-1143).

[8] Kaufman, S., Rosset, S., Perlich, C., & Stitelman, O. (2012). Leakage in data mining: Formulation, detection, and avoidance. *ACM Transactions on Knowledge Discovery from Data*, *6*(4), 15. <https://doi.org/10.1145/2382577.2382579>

[9] Tüfekci, P. (2014). Prediction of full load electrical power output of a base load operated combined cycle power plant using machine learning methods. *International Journal of Electrical Power & Energy Systems*, *60*, 126-140. <https://doi.org/10.1016/j.ijepes.2014.02.027>

[10] Tsanas, A., & Xifara, A. (2012). Accurate quantitative estimation of energy performance of residential buildings using statistical machine learning tools. *Energy and Buildings*, *49*, 560-567. <https://doi.org/10.1016/j.enbuild.2012.03.003>

[11] Brooks, T. F., Pope, D. S., & Marcolini, M. A. (1989). *Airfoil Self-Noise and Prediction (NASA Reference Publication 1218)*. NASA.

[12] Kaya, H., Tüfekci, P., & Uzun, E. (2019). Predicting CO and NOx emissions from gas turbines: Novel data and a benchmark PEMS. *Turkish Journal of Electrical Engineering and Computer Sciences*, *27*(6), 4783-4796. <https://doi.org/10.3906/elk-1807-87>

[13] Hamidieh, K. (2018). A data-driven statistical model for predicting the critical temperature of a superconductor. *Computational Materials Science*, *154*, 346-354. <https://doi.org/10.1016/j.commatsci.2018.07.052>

[14] Imran, A. A., Rahim, M. S., & Ahmed, T. (2020). *Productivity Prediction of Garment Employees [Dataset]. UCI Machine Learning Repository, ID 597*. <https://archive.ics.uci.edu/dataset/597>

[15] Cortez, P., Cerdeira, A., Almeida, F., Matos, T., & Reis, J. (2009). Modeling wine preferences by data mining from physicochemical properties. *Decision Support Systems*, *47*(4), 547-553. <https://doi.org/10.1016/j.dss.2009.05.016>